# 02 – Entrenamiento YOLOv11

---

## Objetivos
- Fine-tuning de YOLO11 preentrenado sobre el dataset de fracturas óseas utilizando las imágenes originales (RAW).
- Evaluar el desempeño de YOLO11 sin aplicar técnicas externas de preprocesamiento.
- Registrar las métricas obtenidas durante el entrenamiento para su posterior análisis y comparación con los resultados reportados para YOLOv10.

```markdown
- 02 – Entrenamiento YOLOv11
    - EXPERIMENTO 1
        - YOLO11
        - Dataset RAW
        - Sin CLAHE
        - Sin Unsharp Masking
        - Sin preprocessing externo
        - Métricas
```

In [ ]:
import sys
from pathlib import Path

_NB_DIR = Path('__file__' if '__file__' in dir() else '.').resolve()
ROOT = _NB_DIR if (_NB_DIR / 'src').exists() else _NB_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultralytics import YOLO
import yaml
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


PyTorch version: 2.2.2+cpu
CUDA disponible: False


## 1. Leer dataset descargado desde Roboflow

In [2]:
# Apuntar al dataset original (RAW), sin preprocesamiento adicional
DATA_YAML = ROOT / 'data' / 'bone-fracture-detection-daoon-1' / 'data.yaml'
assert DATA_YAML.exists(), 'dataset no encontrado, revisá la carpeta data'
print(f'Dataset YAML: {DATA_YAML}')


Dataset YAML: C:\Users\lbritez\Desktop\CEIA\vpc_II\TP\VPCII_TP_CEIA\data\bone-fracture-detection-daoon-1\data.yaml


## 2. Configuración del experimento

In [3]:
with open(ROOT / 'configs' / 'yolov11.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Configuración cargada:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

Configuración cargada:
  model_weights: yolo11m.pt
  epochs: 100
  imgsz: 640
  batch: 16
  lr0: 0.01
  lrf: 0.001
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3.0
  patience: 30
  seed: 42
  hsv_h: 0.0
  hsv_s: 0.0
  hsv_v: 0.3
  degrees: 5.0
  translate: 0.1
  scale: 0.5
  flipud: 0.0
  fliplr: 0.5
  mosaic: 0.5
  mixup: 0.0
  copy_paste: 0.0
  conf: 0.25
  iou: 0.6


## 3. Carga del modelo preentrenado

In [4]:
model = YOLO(cfg['model_weights'])
print(f'Modelo cargado: {cfg["model_weights"]}')
print(model.info())

Modelo cargado: yolo11m.pt
YOLO11m summary: 231 layers, 20,114,688 parameters, 0 gradients, 68.6 GFLOPs
(231, 20114688, 0, 68.643072)


## 4. Fine-tuning

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=cfg['epochs'],
    imgsz=cfg['imgsz'],
    batch=cfg['batch'],
    workers=cfg['workers'],
    cache=cfg['cache'],
    patience=cfg['patience'],
    lr0=cfg['lr0'],
    lrf=cfg['lrf'],
    momentum=cfg['momentum'],
    weight_decay=cfg['weight_decay'],

    # --- Data Augmentation ---
    hsv_h=cfg.get('hsv_h', 0.0),
    hsv_s=cfg.get('hsv_s', 0.0),
    hsv_v=cfg.get('hsv_v', 0.0),
    flipud=cfg.get('flipud', 0.0),
    fliplr=cfg.get('fliplr', 0.5),
    mosaic=cfg.get('mosaic', 0.0),
    mixup=cfg.get('mixup', 0.0),
    
    # --- Control experimental y reproducibilidad ---
    seed=cfg.get('seed', 42),          # Fijar semilla para comparación justa
    deterministic=True,                # Forzar reproducibilidad en PyTorch/CUDA
    plots=True,                        # Generar curvas PR, F1-Confidence y matrices de confusión
    save=True,                         # Guardar best.pt y last.pt
    val=True,                          # Validar al final de cada época
    save_period=-1,                    # Guardar solo mejor/último (o poner N si quieres checkpoints)
    patience=cfg.get('patience', 50),  # Early stopping si no mejora

    # --- Output & Logging ---
    project=str(ROOT / 'results'),
    name=cfg.get('exp_name', 'exp1_yolo11_raw'),  # Identificador claro del experimento
    exist_ok=False,                    # False crea exp1, exp1_2, etc., sin borrar resultados previos
    device=0 if torch.cuda.is_available() else 'cpu',
)

New https://pypi.org/project/ultralytics/8.4.120 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.117  Python-3.12.13 torch-2.2.2+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\lbritez\Desktop\CEIA\vpc_II\TP\VPCII_TP_CEIA\data\bone-fracture-detection-daoon-1\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0,

## 5. Evaluación rápida en validación

In [ ]:
## 5. Evaluación final en el conjunto de Test (Hold-out test set)

import pandas as pd
from pathlib import Path
from ultralytics import YOLO

# Cargar el mejor checkpoint obtenido
best_model_path = Path(results.save_dir) / 'weights' / 'best.pt'
eval_model = YOLO(str(best_model_path))

# Ejecutar evaluación final sobre el split 'test'
test_results = eval_model.val(
    data=str(DATA_YAML),
    split='test',             # Evaluación sobre datos nunca vistos durante el entrenamiento
    imgsz=cfg['imgsz'],
    batch=cfg['batch'],
    conf=cfg['conf'],
    iou=cfg['iou'],
    plots=True,
    save_json=True            # Útil para análisis posterior de predicciones
)

# 1. Métricas Globales
print("\n" + "="*50)
print("         MÉTRICAS GLOBALES EN TEST (YOLO11)")
print("="*50)
print(f"mAP@0.5:        {test_results.box.map50:.4f}")
print(f"mAP@0.5:0.95:   {test_results.box.map:.4f}")
print(f"Precision (P):  {test_results.box.mp:.4f}")
print(f"Recall (R):     {test_results.box.mr:.4f}")

# 2. Desglose por Clase (Evaluación de robustez ante desbalance)
print("\n" + "="*50)
print("        DESGLOSE POR CLASE EN TEST")
print("="*50)

class_metrics = []
for idx, cls_name in test_results.names.items():
    p = test_results.box.p[idx]
    r = test_results.box.r[idx]
    f1 = (2 * p * r) / (p + r + 1e-16)
    map50 = test_results.box.map50s[idx]
    map50_95 = test_results.box.maps[idx]
    
    class_metrics.append({
        'Clase': cls_name,
        'Precision': f"{p:.4f}",
        'Recall': f"{r:.4f}",
        'F1-Score': f"{f1:.4f}",
        'mAP@0.5': f"{map50:.4f}",
        'mAP@0.5:0.95': f"{map50_95:.4f}"
    })

df_test_classes = pd.DataFrame(class_metrics)
print(df_test_classes.to_string(index=False))

# Guardar métricas a CSV para reportes/gráficos
csv_output_path = Path(results.save_dir) / 'test_metrics_per_class.csv'
df_test_classes.to_csv(csv_output_path, index=False)
print(f"\nMétricas por clase guardadas en: {csv_output_path}")

# 3. Latencia y Tiempos de Inferencia
speed = test_results.speed
preprocess_t = speed.get('preprocess', 0.0)
inference_t = speed.get('inference', 0.0)
postprocess_t = speed.get('postprocess', 0.0)
total_t = preprocess_t + inference_t + postprocess_t
fps = 1000.0 / total_t if total_t > 0 else 0

print("\n" + "="*50)
print("       PERFIL DE VELOCIDAD / LATENCIA (TEST)")
print("="*50)
print(f"Pre-process:    {preprocess_t:.2f} ms")
print(f"Inference:      {inference_t:.2f} ms")
print(f"Post (NMS):     {postprocess_t:.2f} ms")
print(f"Latencia Total: {total_t:.2f} ms ({fps:.2f} FPS)")
print("="*50)

---
## Notas del experimento

*(Completar tras el entrenamiento)*

- **Variante de modelo usada:** ...
- **Épocas entrenadas / early stopping:** ...
- **mAP@0.5 final:** ...
- **Observaciones:** ...